In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from emukit.core import ParameterSpace, ContinuousParameter, DiscreteParameter
from emukit.core.initial_designs.random_design import RandomDesign
from emukit.core.initial_designs.latin_design import LatinDesign

In [2]:
def x_normalizer(X, var_array):
    
    def max_min_scaler(x, x_max, x_min):
        return (x-x_min)/(x_max-x_min)
    x_norm = []
    for x in (X):
           x_norm.append([max_min_scaler(x[i], 
                                         max(var_array[i]), 
                                         min(var_array[i])) for i in range(len(x))])
            
    return x_norm

def x_denormalizer(x_norm, var_array):
    
    def max_min_rescaler(x, x_max, x_min):
        return x*(x_max-x_min)+x_min
    x_original = []
    for x in (x_norm):
           x_original.append([max_min_rescaler(x[i], 
                                         max(var_array[i]), 
                                         min(var_array[i])) for i in range(len(x))])
            
    return x_original

def get_closest_value(given_value, array_list):
    absolute_difference_function = lambda list_value : abs(list_value - given_value)
    closest_value = min(array_list, key=absolute_difference_function)
    return closest_value
    
def get_closest_array(suggested_x, var_list):
    modified_array = []
    for x in suggested_x:
        modified_array.append([get_closest_value(x[i], var_list[i]) for i in range(len(x))])
    return np.array(modified_array)

In [3]:
# Define the variable ranges
spinspeed_min, spinspeed_max, spinspeed_step = [500, 5000, 500] ## Unit: rpm
spinspeed_var = np.arange(spinspeed_min, spinspeed_max+spinspeed_step, spinspeed_step) 
spinspeed_num = len(spinspeed_var)

concentration_min, concentration_max, concentration_step = [0.1, 1.5, 0.1] ## Unit: mol/L
concentration_var = np.arange(concentration_min, concentration_max+concentration_step, concentration_step)
concentration_num = len(concentration_var)

annealingtemp_min, annealingtemp_max, annealingtemp_step = [100, 300, 10] # Unit: degC
annealingtemp_var = np.arange(annealingtemp_min, annealingtemp_max+annealingtemp_step, annealingtemp_step)
annealingtemp_num = len(annealingtemp_var)

solvent_ratio_min, solvent_ratio_max, solvent_ratio_step = [1, 4, 1]  # Unitless, DMF:DMSO = 1:1 to 4:1
solvent_ratio_var = np.arange(solvent_ratio_min, solvent_ratio_max+solvent_ratio_step, solvent_ratio_step)
solvent_ratio_num = len(solvent_ratio_var)

solute_ratio_min, solute_ratio_max, solute_ratio_step = [2.8, 3.2, 0.1]  # Unitless, RbI:BiI3 = 2.8:1 to 3.2:1
solute_ratio_var = np.arange(solute_ratio_min, solute_ratio_max+solute_ratio_step, solute_ratio_step)
solute_ratio_num = len(solute_ratio_var)

var_array = [spinspeed_var, concentration_var, 
             annealingtemp_var, solvent_ratio_var, solute_ratio_var]
x_labels = ['Spin Speed [rpm]',  
            'Precursor Concentration [mol/L]', 
            'Annealing Temperature [degC]',
            'Solvent Ratio',
            'Solute Ratio']

random_state = np.random.RandomState(42)
parameter_space = ParameterSpace([ContinuousParameter('x1', 0, 1),
                                 ContinuousParameter('x2', 0, 1),
                                 ContinuousParameter('x3', 0, 1),
                                 ContinuousParameter('x4', 0, 1),
                                 ContinuousParameter('x5', 0, 1)
                                 ])

# parameter_space = ParameterSpace([DiscreteParameter('x1', np.linspace(0,1, 51)),
#                                  DiscreteParameter('x2', np.linspace(0,1, 51)),
#                                  DiscreteParameter('x3', np.linspace(0,1, 51)),
#                                  DiscreteParameter('x4', np.linspace(0,1, 51)),
#                                  DiscreteParameter('x5', np.linspace(0,1, 51)),
#                                  ])
    

# Generate initial samples with latin hypercube
design = LatinDesign(parameter_space)
x_init = design.get_samples(12)
x_init_original = get_closest_array(x_denormalizer(x_init, var_array),var_array)

df = pd.DataFrame(x_init_original, columns = x_labels)
df_cols = x_labels
df.to_csv("/Users/shengfang/Desktop/TRI/Rb3BiI6/BO_samples_12_latin.csv", index=True)
df

,Spin Speed [rpm],Precursor Concentration [mol/L],Annealing Temperature [degC],Solvent Ratio,Solute Ratio
0,5000.0,0.6,290.0,2.0,3.0
1,3500.0,1.1,220.0,2.0,3.3
2,2500.0,0.7,210.0,1.0,3.1
3,2000.0,0.5,240.0,4.0,3.2
4,3000.0,0.9,140.0,1.0,3.1
5,1500.0,1.0,160.0,2.0,2.8
6,3500.0,0.3,260.0,3.0,2.9
7,2000.0,0.4,110.0,2.0,3.2
8,1000.0,1.2,270.0,3.0,3.2
9,4000.0,1.3,170.0,3.0,2.9
